# Build HyperKvasir metadata + image manifest

Prep pipeline for the HyperKvasir image classification dataset.

## Config

In [1]:
import os
from pathlib import Path
import pandas as pd

from datasets import load_dataset, get_dataset_infos


In [2]:
HF_DATASET = "sahilur/hyper-kvasir-labeled-images"  # includes train/val/test splits and 23 labels
SPLITS = ["train", "validation", "test"]
MAX_SAMPLES_PER_SPLIT = None  # set to an int for a quick smoke test (e.g., 50)

OUT_ROOT = Path("./out")
IMAGES_DIR = OUT_ROOT / "images"
META_DIR = OUT_ROOT / "metadata"
MANIFEST_DIR = OUT_ROOT / "manifests"

for d in [IMAGES_DIR, META_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_META_CSV = META_DIR / "metadata_raw.csv"
ENRICHED_META_CSV = META_DIR / "metadata_enriched.csv"
IMAGE_MANIFEST_CSV = MANIFEST_DIR / "image_manifest.csv"
LABEL_MAP_CSV = META_DIR / "label_map.csv"


In [3]:
info = get_dataset_infos(HF_DATASET)["default"]
label_names = info.features["label"].names
print("Dataset:", HF_DATASET)
print("Splits:", SPLITS)
print("Labels:", label_names)
print("Label count:", len(label_names))


Dataset: sahilur/hyper-kvasir-labeled-images
Splits: ['train', 'validation', 'test']
Labels: ['barretts', 'barretts-short-segment', 'bbps-0-1', 'bbps-2-3', 'cecum', 'dyed-lifted-polyps', 'dyed-resection-margins', 'esophagitis-a', 'esophagitis-b-d', 'hemorrhoids', 'ileum', 'impacted-stool', 'polyps', 'pylorus', 'retroflex-rectum', 'retroflex-stomach', 'ulcerative-colitis-grade-0-1', 'ulcerative-colitis-grade-1', 'ulcerative-colitis-grade-1-2', 'ulcerative-colitis-grade-2', 'ulcerative-colitis-grade-2-3', 'ulcerative-colitis-grade-3', 'z-line']
Label count: 23


In [4]:
def safe_img_id(img, split: str, idx: int) -> str:
    fname = getattr(img, "filename", "") or ""
    if fname:
        return Path(fname).stem
    return f"{split}_{idx:06d}"


def save_image(img, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.save(path, format="JPEG")


manifest_rows = []
meta_rows = []

for split in SPLITS:
    print(f"\nProcessing split: {split}")
    ds = load_dataset(HF_DATASET, split=split)
    if MAX_SAMPLES_PER_SPLIT is not None:
        max_n = min(MAX_SAMPLES_PER_SPLIT, len(ds))
        ds = ds.select(range(max_n))
        print(f"  Truncated to {max_n} samples for smoke test")

    for idx, ex in enumerate(ds):
        img = ex["image"]
        label_id = int(ex["label"])
        label_name = label_names[label_id]
        img_id = safe_img_id(img, split, idx)

        out_path = IMAGES_DIR / split / f"{img_id}.jpg"
        if not out_path.exists():
            save_image(img, out_path)

        manifest_rows.append({
            "split": split,
            "img_id": img_id,
            "image_path": str(out_path),
            "exists": out_path.exists(),
        })
        meta_rows.append({
            "split": split,
            "img_id": img_id,
            "image_path": str(out_path),
            "label_id": label_id,
            "label_name": label_name,
            "orig_height": getattr(img, "height", None),
            "orig_width": getattr(img, "width", None),
        })

manifest = pd.DataFrame(manifest_rows)
meta = pd.DataFrame(meta_rows)

manifest.to_csv(IMAGE_MANIFEST_CSV, index=False)
meta.to_csv(RAW_META_CSV, index=False)
meta.to_csv(ENRICHED_META_CSV, index=False)
pd.DataFrame({"label_id": list(range(len(label_names))), "label_name": label_names}).to_csv(LABEL_MAP_CSV, index=False)

print("Saved:", IMAGE_MANIFEST_CSV, "rows:", len(manifest))
print("Saved:", RAW_META_CSV, "rows:", len(meta))



Processing split: train



Processing split: validation



Processing split: test


Saved: out/manifests/image_manifest.csv rows: 10662
Saved: out/metadata/metadata_raw.csv rows: 10662


In [5]:
print("Records per split:")
print(meta.groupby("split").size())

print("Class distribution (overall):")
print(meta["label_name"].value_counts())


Records per split:
split
test          1065
train         8528
validation    1069
dtype: int64
Class distribution (overall):
label_name
bbps-2-3                        1148
polyps                          1028
cecum                           1009
dyed-lifted-polyps              1002
pylorus                          999
dyed-resection-margins           989
z-line                           932
retroflex-stomach                764
bbps-0-1                         646
ulcerative-colitis-grade-2       443
esophagitis-a                    403
retroflex-rectum                 391
esophagitis-b-d                  260
ulcerative-colitis-grade-1       201
ulcerative-colitis-grade-3       133
impacted-stool                   131
barretts-short-segment            53
barretts                          41
ulcerative-colitis-grade-0-1      35
ulcerative-colitis-grade-2-3      28
ulcerative-colitis-grade-1-2      11
ileum                              9
hemorrhoids                        6
Name: count, 